## this notebook includes the method of handling missing values by using a technique where we make a new column for every column containing missing values.

It is based on the same edge of thinking where we try to tell model that there exists missing value in the data.<br>
we tell the model about the missingness inthe data by creating columns for each column containing missing values.

thus, the technique is called missing indicator. we are indicating the model, hey! here the data is missing.<br> and fortuately, this method relly improves the model on some occasions.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import MissingIndicator,SimpleImputer

In [2]:
df = pd.read_csv('titanic_toy.csv',usecols=['Age','Fare','Survived'])

In [3]:
df.isnull().mean() * 100

Age         19.865320
Fare         5.050505
Survived     0.000000
dtype: float64

In [4]:
df.head()

,Age,Fare,Survived
0,22.0,7.2500,0
1,38.0,71.2833,1
2,26.0,7.9250,1
3,35.0,53.1000,1
4,35.0,8.0500,0


In [5]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
X_train.head()

,Age,Fare
331,45.5,28.5000
733,23.0,13.0000
382,32.0,7.9250
704,26.0,7.8542
813,6.0,31.2750


# Observing model performance with out missing indicator technique

In [8]:
si = SimpleImputer()
X_train_trf = si.fit_transform(X_train)
X_test_trf = si.transform(X_test)

In [9]:
X_train_trf

array([[ 45.5       ,  28.5       ],
       [ 23.        ,  13.        ],
       [ 32.        ,   7.925     ],
       ...,
       [ 41.        ,  32.51778772],
       [ 14.        , 120.        ],
       [ 21.        ,  77.2875    ]], shape=(712, 2))

In [10]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()

clf.fit(X_train_trf,y_train)

y_pred = clf.predict(X_test_trf)

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6480446927374302

# Observing model performance using  missing indicator technique

In [11]:
mi = MissingIndicator()

mi.fit(X_train)

MissingIndicator()

In [12]:
mi.features_

array([0, 1])

In [13]:
X_train_missing = mi.transform(X_train)

In [14]:
X_train_missing

array([[False, False],
       [False, False],
       [False, False],
       ...,
       [False,  True],
       [False, False],
       [False, False]], shape=(712, 2))

In [15]:
X_test_missing = mi.transform(X_test)

In [16]:
X_test_missing[0]

array([ True, False])

In [17]:
X_train['Age_NA'] = X_train_missing[:,0]
X_train['Fare_NA'] = X_train_missing[:,1]


In [18]:
X_train

,Age,Fare,Age_NA,Fare_NA
331,45.5,28.5000,False,False
733,23.0,13.0000,False,False
382,32.0,7.9250,False,False
704,26.0,7.8542,False,False
813,6.0,31.2750,False,False
...,...,...,...,...
106,21.0,7.6500,False,False
270,NaN,31.0000,True,False
860,41.0,NaN,False,True
435,14.0,120.0000,False,False


In [19]:
X_test['Age_NA'] = X_test_missing[:,0]
X_test['Fare_NA'] = X_test_missing[:,1]


In [20]:
X_test

,Age,Fare,Age_NA,Fare_NA
709,NaN,15.2458,True,False
439,31.0,10.5000,False,False
840,20.0,7.9250,False,False
720,6.0,33.0000,False,False
39,14.0,11.2417,False,False
...,...,...,...,...
433,17.0,7.1250,False,False
773,NaN,7.2250,True,False
25,38.0,31.3875,False,False
84,17.0,10.5000,False,False


In [27]:
si = SimpleImputer() # by deafault, the missing values are imputed with mean value

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [28]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()

clf.fit(X_train_trf2,y_train)

y_pred = clf.predict(X_test_trf2)

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6312849162011173

# implementing missing indicator technique using sklearn.

In [33]:
si = SimpleImputer(add_indicator=True) # to add missing indicator technique,we simply use 'add_indicator' parameter and set it to true

In [34]:
X_train_tfr = si.fit_transform(X_train)

In [35]:
X_test_tfr = si.transform(X_test)

In [36]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()

clf.fit(X_train_tfr,y_train)

y_pred = clf.predict(X_test_tfr)

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6312849162011173